# Phase 4-A｜新 TSM Development Snapshot 研究授權

這份 Notebook **只簽署新的 Research-only 授權**，由研究負責人本人在 Colab 執行。原始 Holdout lock、舊授權 `SMOKE-0001`、Formal Gates 和成本設定不會變動。

授權範圍：TSM，SMA 5/20，2024-06-03～2024-06-28（2024-05-29 起 warm-up），四組成本情境（BASE／LOW／STRESS／CAT_STRESS），**最多一個研究批次**，2026-11-30 23:59:59（Asia/Taipei）到期。此簽核**不會執行回測、不會使用 Holdout、不會消耗任何研究額度，也不會合併 PR**。

- 新 Snapshot：`TSM_DEV_1M_20240529_20240628_18350b3adba8e28a7e5e`
- Source Manifest SHA：`8b6fc838b72ae3015f6eb418c3a77115e53a86f185e05c2249e9dcd58192cfc5`
- Lineage SHA：`3455f2b3d39f4fd05dda7ca9fed472c9ff0bcbf345f89f177e78eccbe5fd92a4`
- Snapshot producer Core SHA：`b16179a7cc077c36046574769a32c367fa476cae`
- Agent PR： https://github.com/hh4832/strategy-research-agent/pull/5

**操作**：Colab 左側 Secrets 設定 `GITHUB_TOKEN`（僅授權 `hh4832/strategy-research-agent` Contents Read/Write，允許 Notebook 存取）；然後執行「Run All」。執行 Notebook 是研究負責人本人的簽核動作，不需輸入長 SHA。若 token 無法證明 GitHub 登入為 `hh4832`、提案 pin 不符合或資料 preflight 不成立，會在簽核前停止。

輸出：在原 PR #5 分支提交唯一新增授權紀錄與兩個新的版本化設定檔。簽核完成後只執行**非消耗性** governance preflight；實際回測需要另外發動。


In [ ]:
from pathlib import Path
import json, os, re, shutil, subprocess, sys, tempfile, urllib.request
from datetime import datetime, timezone

REPO = "hh4832/strategy-research-agent"
BRANCH = "feature/phase4a-tsm-research-only"
APPROVER = "hh4832"
APPROVAL_REF = "https://github.com/hh4832/strategy-research-agent/pull/5"
WORK = Path("/content/phase4a_tsm_dev_signoff")
REPORT_DIR = Path("/content/drive/MyDrive/00Quant_Research/phase4a_preparation_reports/20261106T105801_339072Z")
SOURCE_SNAPSHOT = Path("/content/drive/MyDrive/00Quant_Research/data/development_snapshots/TSM_DEV_1M_20240529_20240628_18350b3adba8e28a7e5e")
ID = "RA-IBKR_TSM-SMA520_DEV-18350b3adba8e28a7e5e"
SNAPSHOT_ID = "TSM_DEV_1M_20240529_20240628_18350b3adba8e28a7e5e"
CORE_SHA = "b16179a7cc077c36046574769a32c367fa476cae"
MANIFEST_SHA = "8b6fc838b72ae3015f6eb418c3a77115e53a86f185e05c2249e9dcd58192cfc5"
LINEAGE_SHA = "3455f2b3d39f4fd05dda7ca9fed472c9ff0bcbf345f89f177e78eccbe5fd92a4"
CONFIG_SHA = "b6ac297faec9e289b422086b06eccd22d65afed5c38525a9c3eb6247e15bbfe9"
SOURCE_PROPOSAL = REPORT_DIR / "new_unsigned_research_proposal/research_authorization.proposed.json"
SOURCE_CONFIG = REPORT_DIR / "new_unsigned_research_proposal/research_config.proposed.json"
SOURCE_PREFLIGHT = REPORT_DIR / "new_unsigned_research_proposal/data_preflight.json"
NEW_PROPOSAL = "governance/proposals/RA-IBKR_TSM-SMA520_DEV-18350b3adba8e28a7e5e.json"
NEW_CONFIG = "configs/research_only/tsm_sma_5_20_smoke_2024_06_dev_18350b3adba8e28a7e5e.json"
BOOK = "governance/research_authorizations.jsonl"
USES = "registry/research_authorization_uses.jsonl"
GATES = "governance/market_readiness.json"

assert sys.version_info >= (3, 11), "Python 3.11 or newer required"
def run(*cmd, cwd=None, env=None):
    return subprocess.run(cmd, cwd=cwd, env=env, text=True, capture_output=True, check=True)
print("簽核目標：", ID)
print("策略：TSM 15m SMA 5/20；四情境／一批次；不執行回測")


In [ ]:
from google.colab import drive, userdata
drive.mount("/content/drive")
TOKEN = userdata.get("GITHUB_TOKEN")
assert TOKEN, "請在 Colab Secrets 設定 GITHUB_TOKEN 並開啟 notebook access"
request = urllib.request.Request(
    "https://api.github.com/user",
    headers={"Authorization": "Bearer " + TOKEN, "Accept": "application/vnd.github+json"},
)
with urllib.request.urlopen(request, timeout=25) as response:
    github_login = json.load(response)["login"]
assert github_login.lower() == APPROVER.lower(), "GitHub token 所屬帳號不是研究負責人 hh4832"
print("GitHub 研究負責人身分驗證：PASS —", github_login)
assert SOURCE_PROPOSAL.is_file() and SOURCE_CONFIG.is_file() and SOURCE_PREFLIGHT.is_file()
assert (SOURCE_SNAPSHOT / "manifest.json").is_file() and (SOURCE_SNAPSHOT / "lineage.json").is_file()
proposal = json.loads(SOURCE_PROPOSAL.read_text())
cfg = json.loads(SOURCE_CONFIG.read_text())
data_preflight = json.loads(SOURCE_PREFLIGHT.read_text())
assert data_preflight["status"] == "PASS" and all(
    v == "PASS" for v in data_preflight["checks"].values()
), "真實資料 preflight 未全部通過"
assert proposal["authorization_id"] == cfg["authorization_id"] == ID
assert proposal["dataset"] == cfg["dataset"]
assert proposal["dataset"]["snapshot_id"] == SNAPSHOT_ID
assert proposal["dataset"]["producer_commit_sha"] == CORE_SHA
assert cfg["core"]["phase4a_sha"] == CORE_SHA
assert cfg["snapshot_binding"]["manifest_sha256"] == MANIFEST_SHA
assert cfg["snapshot_binding"]["snapshot_pin"]["lineage_sha256"] == LINEAGE_SHA
assert cfg["provenance"]["config_sha256"] == CONFIG_SHA
assert proposal["scope"] == "RESEARCH_ONLY_EXPLORATORY"
assert proposal["max_batches"] == 1
assert proposal["strategy"]["parameters"] == {"fast": 5, "slow": 20}
assert proposal["costs"]["scenarios"] == ["BASE", "LOW", "STRESS", "CAT_STRESS"]
assert proposal["holdout"]["blindness"] == "BLINDNESS_UNVERIFIED"
assert proposal["formal_gate_pin"]["market_status"] == "NO_GO"
assert proposal["expires_at"] == "2026-11-30T23:59:59+08:00"
assert data_preflight["native_rows"] == 21119
assert data_preflight["conservative_buckets"] == 1408
print("已核准新 Snapshot 資料／研究範圍：PASS")
print("不授權正式交易、不授權 Holdout 使用、不授權參數搜尋")


In [ ]:
# 新的乾淨 clone，不刪除既有工作區；同名資料夾存在則停下。
assert not WORK.exists(), f"工作路徑已存在，請改用全新 Colab runtime：{WORK}"
run("git", "clone", "--quiet", "--single-branch", "--branch", BRANCH,
    "https://github.com/" + REPO + ".git", str(WORK))
assert run("git", "branch", "--show-current", cwd=WORK).stdout.strip() == BRANCH
assert run("git", "status", "--porcelain", cwd=WORK).stdout.strip() == ""
BASE_HEAD = run("git", "rev-parse", "HEAD", cwd=WORK).stdout.strip()
assert re.fullmatch(r"[0-9a-f]{40}", BASE_HEAD)
assert run("git", "remote", "get-url", "origin", cwd=WORK).stdout.strip().endswith(
    "hh4832/strategy-research-agent.git"
)
sys.path.insert(0, str(WORK / "src"))
run(sys.executable, "-m", "pip", "install", "--quiet", "-e", str(WORK) + "[research]")
from strategy_agent.governance.research_authorization import ResearchAuthorizationBook
from strategy_agent.governance.research_authorization_admin import approve
from strategy_agent.research_only.runner import ResearchOnlyRunner, canonical_sha
from strategy_agent.registry.chain import ChainedLog
from jsonschema import Draft202012Validator

assert canonical_sha(cfg) == CONFIG_SHA, "已產生的 config hash 不符合研究釘選"
schema = json.loads((WORK / "schemas/RESEARCH_AUTHORIZATION_SCHEMA.json").read_text())
assert re.fullmatch(schema["$defs"]["approve"]["properties"]["authorization_id"]["pattern"], ID)
assert not list(Draft202012Validator(schema).iter_errors({
    **{k: v for k,v in proposal.items() if k not in
       ("proposal_status", "proposed_by", "proposed_at", "approval_instructions")},
    "approved_by": APPROVER, "approval_ref": APPROVAL_REF
}))
assert ResearchAuthorizationBook(WORK / BOOK, WORK / "governance/approvers.json").get(ID) is None, (
    "這份授權已存在，不得重複簽署"
)
assert not (WORK / NEW_PROPOSAL).exists() and not (WORK / NEW_CONFIG).exists()
before_book = (WORK / BOOK).read_bytes()
before_uses = (WORK / USES).read_bytes()
before_gates = (WORK / GATES).read_bytes()
before_holdout = (WORK / "governance/holdout_locks.jsonl").read_bytes()
print("簽核前治理與 Schema：PASS；Git HEAD =", BASE_HEAD)
print("下一步：由研究負責人本人執行治理管理工具追加新授權")


In [ ]:
# 執行本儲存格是研究負責人本人授權研究-only 的動作。
# 不需要手動輸入 SHA 或改 JSON，不呼叫 Runner.run / consume。
(WORK / NEW_PROPOSAL).parent.mkdir(parents=True, exist_ok=True)
(WORK / NEW_CONFIG).parent.mkdir(parents=True, exist_ok=True)
shutil.copyfile(SOURCE_PROPOSAL, WORK / NEW_PROPOSAL)
shutil.copyfile(SOURCE_CONFIG, WORK / NEW_CONFIG)
record = approve(
    WORK / BOOK, WORK / "governance/approvers.json",
    proposal, approved_by=APPROVER, approval_ref=APPROVAL_REF,
)
assert record["authorization_id"] == ID and record["approved_by"] == APPROVER
assert (WORK / BOOK).read_bytes().startswith(before_book), "既有授權鏈被改動"
assert (WORK / USES).read_bytes() == before_uses, "舊研究使用額度變更"
assert (WORK / GATES).read_bytes() == before_gates, "Formal Gates 發生變更"
assert (WORK / "governance/holdout_locks.jsonl").read_bytes() == before_holdout
book = ResearchAuthorizationBook(WORK / BOOK, WORK / "governance/approvers.json")
assert book.get(ID)["dataset"]["snapshot_id"] == SNAPSHOT_ID
pf = ResearchOnlyRunner(root=WORK).preflight(WORK / NEW_CONFIG)
print("研究授權 governance preflight =", pf.status, "blockers =", pf.blockers)
assert pf.status == "READY_FOR_CORE_INTEGRATION" and not pf.blockers, (
    "授權已在本次暫存 clone 建立，但 preflight 尚未通過；不得 push"
)
assert run("git", "status", "--porcelain", cwd=WORK).stdout.strip()
changed = run("git", "status", "--porcelain", cwd=WORK).stdout.splitlines()
allowed = {NEW_PROPOSAL, NEW_CONFIG, BOOK}
assert all(x[3:] in allowed for x in changed), "意外檔案變更：" + repr(changed)
print("簽核紀錄與非消耗性 preflight：PASS；尚未 push")


In [ ]:
# 提交到 Agent 既有 PR #5；不合併，不執行回測。
run("git", "add", "--", NEW_PROPOSAL, NEW_CONFIG, BOOK, cwd=WORK)
run("git", "-c", "user.name=" + APPROVER,
    "-c", "user.email=" + APPROVER + "@users.noreply.github.com",
    "commit", "-m", "governance: research lead approves TSM development-only SMA520 research batch",
    cwd=WORK)

# 僅在推送時使用 Colab Secret。Git remote 不儲存 token，輸出不顯示 token。
with tempfile.TemporaryDirectory() as d:
    askpass = Path(d) / "git-askpass.sh"
    askpass.write_text("#!/bin/sh\ncase \"$1\" in *Username*) printf '%s' \"$GIT_USERNAME\";; *) printf '%s' \"$GIT_PASSWORD\";; esac\n")
    askpass.chmod(0o700)
    env = dict(os.environ)
    env.update(GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0",
               GIT_USERNAME="x-access-token", GIT_PASSWORD=TOKEN)
    res = subprocess.run(["git", "push", "origin", "HEAD:" + BRANCH],
                         cwd=WORK, env=env, text=True, capture_output=True)
    if res.returncode != 0:
        raise RuntimeError("GitHub push 未完成；未宣稱授權生效。請先檢查 branch 並避免重複執行。Git exit code: " + str(res.returncode))
    env["GIT_PASSWORD"] = ""
new_commit = run("git", "rev-parse", "HEAD", cwd=WORK).stdout.strip()
assert run("git", "status", "--porcelain", cwd=WORK).stdout.strip() == ""
print("研究負責人 GitHub 授權已推送至既有 Agent PR #5")
print("Commit SHA:", new_commit)
print("授權 ID:", ID)
print("Agent PR:", APPROVAL_REF)
print("治理 preflight:", pf.status)
print("Formal Gate: UNCHANGED / NO_GO")
print("Holdout: NOT ACCESSED")
print("原始研究額度：UNCONSUMED")
print("NEW batch uses: 0 / 1")
print("真實 SMA 回測：NOT RUN（需下一次獨立觸發）")


## 驗收判準

執行成功時會顯示 Git commit SHA，並將以下三份文件提交到 Agent PR #5：

- `governance/proposals/RA-IBKR_TSM-SMA520_DEV-18350b3adba8e28a7e5e.json`（已核對的原始提案）
- `configs/research_only/tsm_sma_5_20_smoke_2024_06_dev_18350b3adba8e28a7e5e.json`（釘選的新研究設定）
- `governance/research_authorizations.jsonl`（經研究負責人執行 admin append-only 簽核）

本 Notebook 故意不呼叫 `ResearchOnlyRunner.run()`。若簽核或 push 失敗，不要重跑簽核 Cell 造成重複紀錄；先檢查 GitHub PR 和執行輸出。
